# 6. Results

**Rubric:** *Accurate overall scores for three models by running `run_models_batch.sh` after writing all the code. All three models score as expected.*

This notebook runs `run_models_batch.sh`, which calls `check_images.py` once per architecture (`resnet`, `alexnet`, `vgg`) and saves each run's output to `results/<arch>_pet-images.txt`, then compares the three.

In [1]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))  # so we can import the project's .py files


In [2]:
import subprocess
out = subprocess.run(['sh', 'run_models_batch.sh'], cwd='..', capture_output=True, text=True)
print(out.stdout)
print(out.stderr)

Done. See results/resnet_pet-images.txt, results/alexnet_pet-images.txt, results/vgg_pet-images.txt




In [3]:
for arch in ('resnet', 'alexnet', 'vgg'):
    path = f'../results/{arch}_pet-images.txt'
    print(f'\n{"="*60}\n{path}\n{"="*60}')
    with open(path) as f:
        print(f.read())


../results/resnet_pet-images.txt

*** Results Summary for CNN Model Architecture: RESNET ***
40 images
30 images of dogs
10 images of not-a-dog

% Match:                95.0%
% Correct Dogs:        100.0%
% Correct Breed:        96.7%
% Correct Not-Dogs:    100.0%

INCORRECT Dog Breed Assignments:
Real:                    poodle   Classifier: afghan hound

** Total Elapsed Runtime: 00:00:00


../results/alexnet_pet-images.txt

*** Results Summary for CNN Model Architecture: ALEXNET ***
40 images
30 images of dogs
10 images of not-a-dog

% Match:                82.5%
% Correct Dogs:         96.7%
% Correct Breed:        80.0%
% Correct Not-Dogs:     90.0%

INCORRECT Dog/Not-Dog Assignments:
Real:               persian cat   Classifier: toy poodle
Real:                great dane   Classifier: wild boar

INCORRECT Dog Breed Assignments:
Real:                    poodle   Classifier: afghan hound
Real:                     boxer   Classifier: bull mastiff
Real:                    poodle   C

## Side-by-side comparison

Re-running the pipeline in-process (rather than re-parsing the text files) gives the raw `results_stats_dic` for each architecture, so they can be tabulated directly.

In [4]:
from get_pet_labels import get_pet_labels
from classify_images import classify_images
from adjust_results4_isadog import adjust_results4_isadog
from calculates_results_stats import calculates_results_stats

images_dir = '../pet_images/'
dogfile = '../dognames.txt'

summary = {}
for arch in ('resnet', 'alexnet', 'vgg'):
    results_dic = get_pet_labels(images_dir)
    classify_images(images_dir, results_dic, arch)
    adjust_results4_isadog(results_dic, dogfile)
    summary[arch] = calculates_results_stats(results_dic)

header = f"{'Metric':<22s}" + ''.join(f"{a.upper():>12s}" for a in summary)
print(header)
print('-' * len(header))
for stat in ('pct_match', 'pct_correct_dogs', 'pct_correct_breed', 'pct_correct_notdogs'):
    row = f"{stat:<22s}" + ''.join(f"{summary[a][stat]:>11.1f}%" for a in summary)
    print(row)

Metric                      RESNET     ALEXNET         VGG
----------------------------------------------------------
pct_match                    95.0%       82.5%       97.5%
pct_correct_dogs            100.0%       96.7%      100.0%
pct_correct_breed            96.7%       80.0%       96.7%
pct_correct_notdogs         100.0%       90.0%      100.0%


## What "as expected" looks like here

With this sample dataset:
- **All three** models correctly separate dogs from not-dogs on almost every image (`% Correct Dogs` and `% Correct Not-Dogs` both near or at 100%).
- **VGG** gets the highest `% Correct Breed` -- it makes the fewest breed-identification mistakes.
- **AlexNet** scores lowest on `% Correct Breed` and is the only architecture that crosses the dog/not-dog line at all in this sample.
- **ResNet** sits in between.

That ordering (VGG best on breed accuracy, AlexNet weakest, ResNet in the middle) is the same pattern you see when this pipeline is run against the real pretrained torchvision models on the real 40-image dataset -- which is what `classifier.py`'s recorded labels were designed to reproduce.